# bronze → silver: 年齢各歳・男女別人口(050008_toukeika_032)

SageMaker Unified Studio の JupyterLab(Python 3 カーネル)で上から順に実行する。
SQL は Athena(ワークグループ `jaws-festa-2026`)で流す。`primary` ワークグループは
結果の保存先が `-an` バケットで `MissingNamespaceHeader` になるので使わない。

先に `landing_to_bronze.ipynb` で bronze に書き出してあること。

| silver テーブル | 役割 | 作り方 |
| --- | --- | --- |
| `silver_nenreikakusai` | 全期間を正しく読める本番用 | ブロックごとに**見出しが一致する定義**の bronze から読む(1980〜2015 は128列定義、2020 は127列定義) |
| `silver_nenreikakusai_128` | 列ずれを見せる比較用 | bronze `_128` を位置で読んだまま型だけ付ける |
| `silver_nenreikakusai_127` | 列ずれを見せる比較用 | bronze `_127` を位置で読んだまま型だけ付ける |

値の扱い(決定済み):

| 値 | silver での扱い |
| --- | --- |
| 人数の列 | `try_cast(... AS bigint)` で long にする |
| `-`(2020ブロックの40か所) | NULL(0にはしない)。件数を記録する |
| マイナスの値 | 推計値としてそのまま残す。件数を記録する |
| `年齢不詳`(2020ブロック) | NULL。2020年は国勢調査の不詳補完値で、年齢不詳が各歳に振り分け済みのため |

手順:

1. 見出しの確認 — bronze の各 CSV の見出しが、使う定義(Glue の列コメント)と一致するか
2. 書き込む前の確認(bronze を読むだけ) — 型変換できない値 / 合計の検算 / キー重複 / 年とブロック / 年の抜け / コードと年月日 / 男女別の3行 / 件数。前年比・検算を飛ばした行・市町村名の変化は記録だけ
3. 書き込み — `DELETE` → `INSERT`(何度実行しても同じ結果になる)
4. 書き込み後の確認 — 件数と NULL の数

確認で1つでも問題があれば、silver には何も書かない。

In [ ]:
# ── 設定 ──────────────────────────────────────────────────────────────────
import os

# True のあいだは確認だけして書き込まない。結果を見てから False にして再実行する。
DRY_RUN = True

REGION = "ap-northeast-1"
WORKGROUP = "jaws-festa-2026"
BUCKET_PREFIX = "jaws-festa-2026"
BRONZE_PREFIX = "akita/bronze/population_by_age/"

BRONZE_DB = '"awsdatacatalog"."jaws_festa_2026_akita"'
SILVER_DB = '"s3tablescatalog/jaws-festa-2026-silver"."akita"'

# 数値にならなくても許す値(NULL にする)。これ以外が出たら止める。
NULL_TOKENS = {"-"}

# 識別列。これ以外は人数なので bigint(Iceberg の long)にする。
STRING_COLUMNS = ["nengetsubi_code", "nengetsubi", "shichoson_code", "shichoson", "danjo"]

In [ ]:
import time
import boto3

session = boto3.Session(profile_name=os.environ.get("AWS_PROFILE"), region_name=REGION)
s3 = session.client("s3")
glue = session.client("glue")
athena = session.client("athena")
account_id = session.client("sts").get_caller_identity()["Account"]
BUCKET = f"{BUCKET_PREFIX}-{account_id}-{REGION}-an"


def run(sql: str, timeout: int = 300) -> list[list[str | None]]:
    """Athena で SQL を1本流して、見出し行を除いた結果を返す。"""
    qid = athena.start_query_execution(QueryString=sql, WorkGroup=WORKGROUP)["QueryExecutionId"]
    deadline = time.time() + timeout
    while True:
        st = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]
        if st["State"] in ("SUCCEEDED", "FAILED", "CANCELLED"):
            break
        if time.time() > deadline:
            athena.stop_query_execution(QueryExecutionId=qid)
            raise TimeoutError(f"{timeout}秒で終わらなかったので止めた: {qid}")
        time.sleep(1)
    if st["State"] != "SUCCEEDED":
        raise RuntimeError(f"{st['State']}: {st.get('StateChangeReason')}\n{sql[:500]}")
    rows = []
    for page in athena.get_paginator("get_query_results").paginate(QueryExecutionId=qid):
        rows += [[c.get("VarCharValue") for c in r["Data"]] for r in page["ResultSet"]["Rows"]]
    return rows[1:] if rows and sql.lstrip().upper().startswith(("SELECT", "WITH")) else rows


def columns_of(table: str) -> list[dict]:
    t = glue.get_table(DatabaseName="jaws_festa_2026_akita", Name=table)["Table"]
    return t["StorageDescriptor"]["Columns"]


COLS_128 = columns_of("bronze_nenreikakusai_128")
COLS_127 = columns_of("bronze_nenreikakusai_127")
NAMES_128 = [c["Name"] for c in COLS_128]
NAMES_127 = [c["Name"] for c in COLS_127]
print(f"bucket: {BUCKET}")
print(f"bronze 定義: 128列 {len(NAMES_128)} / 127列 {len(NAMES_127)}")

## 1. 見出しの確認

bronze のテーブルは列を**位置**で読むので、CSV の見出しと定義がずれていても気づけない。
ここで各 CSV の1行目(見出し)を読み、Glue の列コメント(元の日本語見出し)と突き合わせて、
ブロックごとにどちらの定義が正しいかを決める。どちらとも一致しないブロックがあれば止める。

In [ ]:
import re

def expected_header(cols: list[dict]) -> list[str]:
    # コメントは「年齢不詳(2020ブロックには存在しない)」のように注記付きのことがあるので、括弧の前だけ使う
    return [c.get("Comment", "").split("(")[0] for c in cols]

EXP_128, EXP_127 = expected_header(COLS_128), expected_header(COLS_127)

keys = sorted(
    o["Key"]
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=BRONZE_PREFIX)
    for o in page.get("Contents", [])
    if o["Key"].endswith(".csv")
)
assert keys, f"bronze が空: s3://{BUCKET}/{BRONZE_PREFIX}  先に landing_to_bronze.ipynb を実行する"

BLOCK_DEF = {}  # block -> "128" / "127"
for key in keys:
    block = re.search(r"_(\d{4})\.csv$", key).group(1)
    head = s3.get_object(Bucket=BUCKET, Key=key, Range="bytes=0-65535")["Body"].read()
    # 改行の手前までをデコードする(途中で切ると多バイト文字が割れるため)
    header = head.split(b"\n", 1)[0].rstrip(b"\r").decode("utf-8-sig").split(",")
    header = [h.strip() for h in header]
    if header == EXP_128:
        BLOCK_DEF[block] = "128"
    elif header == EXP_127:
        BLOCK_DEF[block] = "127"
    else:
        diff = next((i for i, (a, b) in enumerate(zip(header, EXP_128)) if a != b), None)
        raise SystemExit(f"{block}: 見出しがどちらの定義とも一致しない(列数 {len(header)}、128列定義と最初に違う位置 {diff})")
    print(f"  {block}: {len(header)}列 -> {BLOCK_DEF[block]}列定義で読む")

OLD_BLOCKS = sorted(b for b, d in BLOCK_DEF.items() if d == "128")
NEW_BLOCKS = sorted(b for b, d in BLOCK_DEF.items() if d == "127")

## 2. 書き込む前の確認

silver に入れる SELECT をここで組み立てて、その結果に対して確認する。

| 確認 | 問題とみなす条件 |
| --- | --- |
| 型変換 | `NULL_TOKENS`(`-`)以外の、数値にならない値がある |
| 合計の検算 | 総数 ≠ 各歳+100歳以上+年齢不詳 / 男女計 ≠ 男+女 / 5歳階級 ≠ 各歳の合計(NULL を含む行は除く) |
| キー重複 | 年月日コード × 市町村コード × 男女別 が2行以上ある |
| 件数 | silver に入る行数と bronze の行数が違う |
| 年とブロック | 行の年が、ファイルのブロックの範囲(そのブロックの先頭年〜次のブロックの前年。最後のブロックは上限なし)に入っていない |
| 年の抜け | 最初の年から最後の年までに、行が1つも無い年がある |
| コードと年月日 | 年月日コードの先頭4桁と、年月日の年が違う |
| 男女別の3行 | 年月日コード × 市町村コードごとに、男女計・男・女が1行ずつそろっていない |

次は問題にせず、記録だけする。

| 記録 | 内容 |
| --- | --- |
| マイナス・`-` の件数 | 型変換の確認で数えたもの |
| 5歳階級の前年比(秋田県・男女計) | いちばん大きく減った・増えたところ。列がずれると2019→2020の90〜94歳が −73% になる。止める基準は値の幅を見てから決める |
| 検算を飛ばした行 | NULL(`-`)を含むため合計の検算をしていない行の数 |
| 市町村名の変化 | 同じ市町村コードで名前が2つ以上あるもの(村→町など) |

これらはすべて bronze を読むだけで、silver には触らない。
県の注記どおり、市町村の合計と県の値は一致しないので、その検算はしない。

In [ ]:
BLOCK = "regexp_extract(\"$path\", '_([0-9]{4})\\.csv$', 1)"

def typed(col: str, src: str | None = None) -> str:
    """bronze の string 列を silver の型にする式。src が None なら NULL。"""
    if col in STRING_COLUMNS:
        return f"{src or col} AS {col}"
    if src is None:
        return f"CAST(NULL AS bigint) AS {col}"
    return f"try_cast(trim({src}) AS bigint) AS {col}"

def in_list(blocks: list[str]) -> str:
    return ", ".join(f"'{b}'" for b in blocks) or "NULL"

# 本番用: ブロックごとに見出しが一致する定義から、列名で対応させて読む
canonical_parts = []
if OLD_BLOCKS:
    canonical_parts.append(
        "SELECT " + ", ".join(typed(c, c) for c in NAMES_128)
        + f" FROM {BRONZE_DB}.bronze_nenreikakusai_128 WHERE {BLOCK} IN ({in_list(OLD_BLOCKS)})"
    )
if NEW_BLOCKS:
    # 127列定義には age_unknown が無い -> NULL(不詳補完値のため)
    canonical_parts.append(
        "SELECT " + ", ".join(typed(c, c if c in NAMES_127 else None) for c in NAMES_128)
        + f" FROM {BRONZE_DB}.bronze_nenreikakusai_127 WHERE {BLOCK} IN ({in_list(NEW_BLOCKS)})"
    )
SELECT_CANONICAL = "\nUNION ALL\n".join(canonical_parts)

# 比較用: bronze を位置で読んだまま型だけ付ける(列ずれをそのまま残す)
SELECT_128 = "SELECT " + ", ".join(typed(c, c) for c in NAMES_128) + f" FROM {BRONZE_DB}.bronze_nenreikakusai_128"
SELECT_127 = "SELECT " + ", ".join(typed(c, c) for c in NAMES_127) + f" FROM {BRONZE_DB}.bronze_nenreikakusai_127"

TARGETS = {
    "silver_nenreikakusai": (SELECT_CANONICAL, NAMES_128),
    "silver_nenreikakusai_128": (SELECT_128, NAMES_128),
    "silver_nenreikakusai_127": (SELECT_127, NAMES_127),
}

In [ ]:
problems = []
report = {}

# ── 型変換できない値 / マイナス(元の bronze の文字列で数える) ──────────────
def raw_value_counts(table: str, names: list[str], where: str = "") -> list[list[str]]:
    nums = [c for c in names if c not in STRING_COLUMNS]
    arr = "ARRAY[" + ", ".join(nums) + "]"
    return run(f"""
        SELECT CASE WHEN try_cast(trim(v) AS bigint) IS NULL THEN coalesce(trim(v), '<NULL>')
                    ELSE 'negative' END AS kind,
               count(*)
        FROM {BRONZE_DB}.{table} CROSS JOIN UNNEST({arr}) AS t(v)
        WHERE {where or 'TRUE'}
          AND (try_cast(trim(v) AS bigint) IS NULL OR try_cast(trim(v) AS bigint) < 0)
        GROUP BY 1 ORDER BY 2 DESC""")

# 本番用の読み方(ブロックごとに正しい定義)で数える
canon_counts = {}
for table, names, blocks in (("bronze_nenreikakusai_128", NAMES_128, OLD_BLOCKS),
                             ("bronze_nenreikakusai_127", NAMES_127, NEW_BLOCKS)):
    if blocks:
        for kind, n in raw_value_counts(table, names, f"{BLOCK} IN ({in_list(blocks)})"):
            canon_counts[kind] = canon_counts.get(kind, 0) + int(n)
report["value_counts"] = canon_counts
print("数値にならない値・マイナスの件数:", canon_counts)
bad = {k: v for k, v in canon_counts.items() if k not in NULL_TOKENS and k != "negative"}
if bad:
    problems.append(f"想定外の数値にならない値: {bad}")

# ── 合計の検算(本番用の SELECT に対して) ────────────────────────────────────
ages = [f"age_{i}" for i in range(100)]
sum_ages = " + ".join(ages + ["age_100_plus", "coalesce(age_unknown, 0)"])
cls_checks = " OR ".join(
    f"age_cls_{k*5}_{k*5+4} <> (" + " + ".join(f"age_{k*5+j}" for j in range(5)) + ")" for k in range(20)
)
(tot_bad, cls_bad), = run(f"""
    SELECT count_if(sosu <> ({sum_ages})),
           count_if({cls_checks})
    FROM ({SELECT_CANONICAL})""")
mf_bad, = run(f"""
    WITH s AS ({SELECT_CANONICAL})
    SELECT count(*)
    FROM s t JOIN s m ON m.nengetsubi_code = t.nengetsubi_code AND m.shichoson_code = t.shichoson_code AND m.danjo = '男（人）'
             JOIN s f ON f.nengetsubi_code = t.nengetsubi_code AND f.shichoson_code = t.shichoson_code AND f.danjo = '女（人）'
    WHERE t.danjo = '男女計（人）' AND t.sosu <> m.sosu + f.sosu""")[0]
report["identity_mismatch"] = {"sosu": int(tot_bad), "age_cls": int(cls_bad), "danjo": int(mf_bad)}
print("合計の検算で合わない行:", report["identity_mismatch"])
if any(report["identity_mismatch"].values()):
    problems.append(f"合計の検算が合わない: {report['identity_mismatch']}")

# ── キー重複 ────────────────────────────────────────────────────────────────
dup, = run(f"""
    SELECT count(*) FROM (
      SELECT nengetsubi_code, shichoson_code, danjo FROM ({SELECT_CANONICAL})
      GROUP BY 1, 2, 3 HAVING count(*) > 1)""")[0]
report["duplicate_keys"] = int(dup)
print("キー重複:", dup)
if int(dup):
    problems.append(f"キー重複が {dup} 件")

# ── bronze のキー列(ブロックごとに正しい定義で読む) ─────────────────────────
key_parts = []
for table, blocks in (("bronze_nenreikakusai_128", OLD_BLOCKS), ("bronze_nenreikakusai_127", NEW_BLOCKS)):
    if blocks:
        key_parts.append(
            f"SELECT {BLOCK} AS blk, nengetsubi_code, nengetsubi, shichoson_code, shichoson, danjo "
            f"FROM {BRONZE_DB}.{table} WHERE {BLOCK} IN ({in_list(blocks)})"
        )
BRONZE_KEYS = "\nUNION ALL\n".join(key_parts)

# ── 年とブロック: 先頭年 <= 年 < 次のブロックの先頭年(最後のブロックは上限なし) ──
all_blocks = sorted(BLOCK_DEF)
upper = {b: (int(all_blocks[i + 1]) if i + 1 < len(all_blocks) else 9999) for i, b in enumerate(all_blocks)}
upper_case = "CASE blk " + " ".join(f"WHEN '{b}' THEN {u}" for b, u in upper.items()) + " END"
(out_of_block,), = run(f"""
    SELECT count_if(NOT (CAST(substr(nengetsubi, 1, 4) AS integer) >= CAST(blk AS integer)
                         AND CAST(substr(nengetsubi, 1, 4) AS integer) < {upper_case}))
    FROM ({BRONZE_KEYS})""")
report["out_of_block_rows"] = int(out_of_block)
print("ブロックの範囲外の年の行:", out_of_block)
if int(out_of_block):
    problems.append(f"ブロックの範囲外の年の行が {out_of_block} 件")

# ── 年の抜け ──────────────────────────────────────────────────────────────
years = sorted(int(y) for (y,) in run(f"SELECT DISTINCT substr(nengetsubi, 1, 4) FROM ({BRONZE_KEYS})"))
missing = sorted(set(range(years[0], years[-1] + 1)) - set(years))
report["years"] = {"first": years[0], "last": years[-1], "missing": missing}
print(f"年: {years[0]}〜{years[-1]}  抜け: {missing or 'なし'}")
if missing:
    problems.append(f"行が無い年がある: {missing}")

# ── 年月日コードと年月日 ────────────────────────────────────────────────────
(code_mismatch,), = run(f"""
    SELECT count_if(substr(nengetsubi_code, 1, 4) <> substr(nengetsubi, 1, 4)) FROM ({BRONZE_KEYS})""")
report["code_date_mismatch"] = int(code_mismatch)
print("年月日コードと年月日が合わない行:", code_mismatch)
if int(code_mismatch):
    problems.append(f"年月日コードと年月日が合わない行が {code_mismatch} 件")

# ── 男女別の3行 ────────────────────────────────────────────────────────────
(not_three,), = run(f"""
    SELECT count(*) FROM (
      SELECT nengetsubi_code, shichoson_code
      FROM ({BRONZE_KEYS})
      GROUP BY 1, 2
      HAVING count(*) <> 3
          OR count(DISTINCT danjo) <> 3
          OR count_if(danjo NOT IN ('男女計（人）', '男（人）', '女（人）')) > 0)""")
report["incomplete_danjo_groups"] = int(not_three)
print("男女別が3行そろっていない組:", not_three)
if int(not_three):
    problems.append(f"男女別が3行そろっていない組が {not_three} 件")

# ── 記録: 検算を飛ばした行(NULL を含む) ────────────────────────────────────
(skip_sosu, skip_cls), = run(f"""
    SELECT count_if((sosu <> ({sum_ages})) IS NULL),
           count_if(({cls_checks}) IS NULL)
    FROM ({SELECT_CANONICAL})""")
report["identity_skipped_rows"] = {"sosu": int(skip_sosu), "age_cls": int(skip_cls)}
print("記録: 検算を飛ばした行(NULL を含む):", report["identity_skipped_rows"])

# ── 記録: 5歳階級の前年比(秋田県・男女計) ──────────────────────────────────
cls_names = [f"age_cls_{k*5}_{k*5+4}" for k in range(20)]
yoy = run(f"""
    WITH s AS (
      SELECT substr(nengetsubi, 1, 4) AS y, {", ".join(cls_names)}
      FROM ({SELECT_CANONICAL})
      WHERE shichoson = '秋田県' AND danjo = '男女計（人）'),
    u AS (
      SELECT y, cls, v FROM s
      CROSS JOIN UNNEST(ARRAY[{", ".join(f"'{c}'" for c in cls_names)}],
                        ARRAY[{", ".join(cls_names)}]) AS t(cls, v))
    SELECT y, cls, round(CAST(v AS double) / lag(v) OVER (PARTITION BY cls ORDER BY y) - 1, 4) AS r
    FROM u ORDER BY y, cls""")
yoy = [(y, c, float(r)) for y, c, r in yoy if r is not None]
lo, hi = min(yoy, key=lambda x: x[2]), max(yoy, key=lambda x: x[2])
report["age_cls_yoy_pref"] = {"min": {"year": lo[0], "cls": lo[1], "ratio": lo[2]},
                              "max": {"year": hi[0], "cls": hi[1], "ratio": hi[2]}}
print(f"記録: 5歳階級の前年比(秋田県) 最小 {lo[0]} {lo[1]} {lo[2]:+.1%} / 最大 {hi[0]} {hi[1]} {hi[2]:+.1%}")

# ── 記録: 同じ市町村コードで名前が変わったもの ────────────────────────────
renamed = run(f"""
    SELECT shichoson_code, array_join(array_sort(array_agg(DISTINCT shichoson)), ' / ')
    FROM ({BRONZE_KEYS})
    GROUP BY 1 HAVING count(DISTINCT shichoson) > 1 ORDER BY 1""")
report["renamed_municipalities"] = {c: n for c, n in renamed}
print("記録: 名前が変わった市町村コード:", report["renamed_municipalities"] or "なし")

# ── 件数 ────────────────────────────────────────────────────────────────────
(bronze_rows,), = run(f"SELECT count(*) FROM {BRONZE_DB}.bronze_nenreikakusai_128")
(canon_rows,), = run(f"SELECT count(*) FROM ({SELECT_CANONICAL})")
report["rows"] = {"bronze": int(bronze_rows), "silver_nenreikakusai": int(canon_rows)}
print("件数:", report["rows"])
if int(bronze_rows) == 0:
    problems.append("bronze が0行")
if bronze_rows != canon_rows:
    problems.append(f"件数が合わない: bronze {bronze_rows} / silver {canon_rows}")

print()
print("確認結果:", "OK" if not problems else "NG")
for p in problems:
    print("  -", p)

## 3. silver に書き込む

確認がすべて通ったときだけ書く。テーブルごとに `DELETE`(全行)→ `INSERT` なので、
何度実行しても同じ結果になる。Iceberg なので各文はそれぞれ一度にコミットされるが、
`DELETE` と `INSERT` の間は一瞬テーブルが空になる。

In [ ]:
if problems:
    raise SystemExit(f"確認に失敗したので silver には書かない: {problems}")

for table, (select_sql, names) in TARGETS.items():
    target = f"{SILVER_DB}.{table}"
    delete_sql = f"DELETE FROM {target}"
    insert_sql = f"INSERT INTO {target} ({', '.join(names)})\n{select_sql}"
    if DRY_RUN:
        print(f"[dry-run] {table}: DELETE -> INSERT ({len(names)}列)")
        continue
    run(delete_sql)
    run(insert_sql)
    print(f"wrote {target}")

## 4. 書き込み後の確認

In [ ]:
if not DRY_RUN:
    for table in TARGETS:
        (n, unknown_null, last_null), = run(f"""
            SELECT count(*), count_if(age_unknown IS NULL), count_if(age_cls_95_99 IS NULL)
            FROM {SILVER_DB}.{table}""") if table != "silver_nenreikakusai_127" else run(f"""
            SELECT count(*), NULL, count_if(age_cls_95_99 IS NULL)
            FROM {SILVER_DB}.{table}""")
        print(f"{table:<28} 行 {n:>5}  age_unknown NULL {unknown_null}  age_cls_95_99 NULL {last_null}")
    (n,), = run(f"SELECT count(*) FROM {SILVER_DB}.silver_nenreikakusai")
    assert n == report["rows"]["bronze"], f"silver_nenreikakusai の行数 {n} が bronze {report['rows']['bronze']} と違う"
    print("書き込み後の確認 OK")

## SQL を確認したいとき

次のセルを実行すると、silver に流す `INSERT` 文をそのまま表示する。
Athena のクエリエディタ(ワークグループ `jaws-festa-2026`)に貼れば同じことができる。

In [ ]:
for table, (select_sql, names) in TARGETS.items():
    print(f"-- {table}")
    print(f"DELETE FROM {SILVER_DB}.{table};")
    print(f"INSERT INTO {SILVER_DB}.{table} ({', '.join(names)})\n{select_sql};\n")